# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the **FAIR²** dataset using the `mlcroissant` library. All record sets, fields, and columns are referenced by their `@id`.

### Dataset Source
The dataset is provided via a Croissant schema URL.

In [ ]:
# Install mlcroissant if not already present
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata and show basic information
dataset = mlc.Dataset(croissant_url)

# Show dataset name and description
print(f"Dataset Title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, their fields, and IDs. 

All entity references use their `@id`.

In [ ]:
# Get all record set @ids
print("Available record sets (by @id):")
for record_set in dataset.metadata.record_sets:
    print(f"- {record_set['@id']}")

# For demonstration, show all fields of each record set by their @id
for record_set in dataset.metadata.record_sets:
    print(f"\n--- Record Set: {record_set['@id']} ---")
    if 'fields' in record_set:
        for field in record_set['fields']:
            # Each field is itself a dictionary, possibly with @id
            print(f"Field @id: {field['@id']}", end=' ')
            # Optionally show name if present
            if 'name' in field:
                print(f"(name: {field['name']})")
            else:
                print()

## 3. Data Extraction
Load data from the primary record set into a DataFrame for analysis.

All entities are referenced by their `@id` as described in the overview above.

In [ ]:
# Extract data from all record sets by their @id
record_set_ids = [r['@id'] for r in dataset.metadata.record_sets]
dataframes = {}

for rec_id in record_set_ids:
    print(f"Loading records from record set '@id': {rec_id}")
    records = list(dataset.records(record_set=rec_id))
    if records:
        dataframes[rec_id] = pd.DataFrame(records)
    else:
        print(f"Warning: No records found for {rec_id}")

# Preview fields and first rows of the main record set
main_record_set = record_set_ids[0] if record_set_ids else None
if main_record_set and main_record_set in dataframes:
    print(f"Columns for main record set {main_record_set}:")
    print(dataframes[main_record_set].columns.tolist())
    display(dataframes[main_record_set].head())
else:
    print("No primary record set with data available for exploration.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records based on criteria, normalizing numeric fields, and grouping. All columns are referenced by their exact `@id`.

In [ ]:
import numpy as np

# Choose the main dataframe and key field @ids for demonstration
df_id = main_record_set

# Automatically detect a likely numeric field by its dtype
if df_id and df_id in dataframes:
    df = dataframes[df_id]
    # Find first numeric field
    candidate_numeric = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            candidate_numeric = col
            break

    if candidate_numeric:
        print(f"Using field @id '{candidate_numeric}' as numeric field for filtering and normalization.")
        # Example threshold (median)
        thresh = df[candidate_numeric].median()
        filtered_df = df[df[candidate_numeric] > thresh]
        print(f"Filtered records with {candidate_numeric} > {thresh:.2f} (median):")
        display(filtered_df.head())

        # Normalize selected field
        filtered_df[f"{candidate_numeric}_normalized"] = (
            filtered_df[candidate_numeric] - filtered_df[candidate_numeric].mean()
        ) / (filtered_df[candidate_numeric].std())
        print(f"Normalized {candidate_numeric} for filtered records:")
        display(filtered_df[[candidate_numeric, f"{candidate_numeric}_normalized"]].head())

        # Try grouping by first non-numeric field
        group_field = None
        for col in df.columns:
            if not pd.api.types.is_numeric_dtype(df[col]) and col != candidate_numeric:
                group_field = col
                break

        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[candidate_numeric].mean().reset_index()
            print(f"Grouped results (mean {candidate_numeric}) by '{group_field}':")
            display(grouped_df.head())
        else:
            print("No non-numeric field found for grouping.")
    else:
        print("No numeric field found in the main DataFrame.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using Matplotlib or Seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df_id and df_id in dataframes and candidate_numeric:
    # Histogram of the numeric column
    plt.figure(figsize=(8,4))
    sns.histplot(df[candidate_numeric].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of '{candidate_numeric}'")
    plt.xlabel(candidate_numeric)
    plt.ylabel('Frequency')
    plt.show()

    # Boxplot by group if group field exists
    if group_field is not None:
        plt.figure(figsize=(10,4))
        sns.boxplot(data=df, x=group_field, y=candidate_numeric)
        plt.title(f"{candidate_numeric} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No suitable data for visualization.")

## 6. Conclusion
In this notebook, you have:
- Loaded the FAIR² mass spectrometry dataset using `mlcroissant`
- Examined record sets and fields by their `@id`
- Loaded records into DataFrames and performed EDA and basic analysis
- Visualized key numeric variables

For deeper analysis, review and utilize the full list of field `@id`s reported in the overview and EDA sections.